# Expression prediction across genes and cell types

This notebook loads a trained sequence-to-expression model, prepares genomic sequences around annotated transcription start sites, pairs each sequence with cell-type metadata, predicts expression, and compares the predictions with held-out measurements.

## Imports

Import filesystem utilities, plotting and table libraries, parallel-processing helpers, and the expression-model API. The final three imports provide the core abstractions used below:

- `SequenceModel` loads and runs the trained model.
- `Genome` retrieves annotated DNA sequences from a reference genome.
- `Condition` and `DescriptionLookup` associate each cell type with its metadata description.

In [1]:
import os
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import polars as pl
from concurrent.futures import ProcessPoolExecutor
from functools import partial
import tqdm
import numpy as np

from gena_expression.conditions import Condition, DescriptionLookup
from gena_expression.genome import Genome
from gena_expression.models import SequenceModel

## Configure paths and inference settings

Define the locations of model code, checkpoint, configuration, tokenizers, genome resources, and metadata. These paths are environment-specific and should be updated when running the notebook elsewhere.

The settings also define the GPU device and the number of DNA tokens fetched upstream of each transcription start site.

In [2]:
# Paths and constants used by later cells

GENA_LM_ROOT = Path("/workspace-SR003.nfs2/estsoi/CAGI5_benchmark") / "GENA_LM" # CHANGE THIS LINE

EXPRESSION_TASK_DIR = GENA_LM_ROOT / "downstream_tasks" / "expression_prediction"

MODEL_CLS = EXPRESSION_TASK_DIR / "expression_model_final.py"
MODEL_CLS = f"{MODEL_CLS}::ExpressionCounts"

MODEL_CHECKPOINT = Path("/workspace-SR003.nfs2/estsoi/CAGI5_benchmark") / "models" / "expression_model" / "pytorch_model.bin" # CHANGE THIS LINE
MODEL_CONFIG = EXPRESSION_TASK_DIR / "inference_example_config.yaml" # CHANGE THIS LINE
DNA_TOKENIZER = GENA_LM_ROOT / "data" / "tokenizers" / "t2t_1000h_multi_32k"
DESCRIPTION_TOKENIZER = "Qwen/Qwen3-Embedding-0.6B"

DATA_DIR = Path("../data").resolve()
#DESCRIPTION_DIR = DATA_DIR / "real_descriptions"
DESCRIPTION_DIR = Path('/workspace-SR003.nfs2/estsoi/scRNA_seq_experiments/GENA_LM/downstream_tasks/expression_prediction/datasets/data/metadata')

GENOME_FASTA = Path("/home/jovyan/.cache/mpramnist/data/Kircher/hg38.fa") # CHANGE THIS LINE
GENOME_ANNOTATION = Path(
    "/workspace-SR003.nfs2/estsoi/scRNA_seq_experiments/GENA_LM/"
    "downstream_tasks/expression_prediction/datasets/data/genomes/hg38/"
    "gencode.v29.primary_assembly.annotation_UCSC_names.gtf"
) # CHANGE THIS LINE

DEVICE = "cuda:5" # CHANGE THIS LINE
os.environ["GENALM_HOME"] = Path('../../../../..').resolve().__str__() # CHANGE THIS LINE

## Load the trained sequence model

Load the expression-prediction model together with its checkpoint, model configuration, DNA tokenizer, and description tokenizer.

The sequence-length settings specify how DNA and metadata are represented at inference time. The model is placed on the selected GPU device.

In [3]:
model = SequenceModel.load(
    model_cls=MODEL_CLS,
    checkpoint=MODEL_CHECKPOINT,
    config=MODEL_CONFIG,
    dna_tokenizer=DNA_TOKENIZER,
    description_tokenizer=DESCRIPTION_TOKENIZER,
    dna_max_seq_len=1024,
    num_before=512,
    desc_max_seq_len=510,
    token_len_for_fetch=15,
    device=DEVICE,
)

/home/jovyan/miniconda3/envs/api/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/workspace-SR003.nfs2/estsoi/API/GENA_LM/downstream_tasks/expression_prediction/api/src/gena_expression/models.py:311: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  with initialize_config_dir(str(config_path.parents[0])):


Using ModernGENA from AIRI-Institute/moderngena-large


Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in ModernBertModel is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `torch_dtype` argument. Example: `model = AutoModel.from_pretrained("openai/whisper-tiny", attn_implementation="flash_attention_2", torch_dtype=torch.float16)`


missing: 0 []
unexpected: 3 ['decoder.bias', 'head.dense.weight', 'head.norm.weight']
mismatched: []
bert dropouts: {'attention_dropout': 0.1, 'embedding_dropout': 0.1, 'mlp_dropout': 0.1}
qwen dropouts: {'attention_dropout': 0.1}
[desc_model] unfrozen transformer blocks: [24, 25, 26, 27] (total blocks=28)
[desc_model] backbone.norm trainable: True (trainable params=1,024)
[desc_model] trainable params: 62,924,800 / 595,776,512
[desc_model] trainable tensors: 45
  - layers.24.self_attn.q_proj.weight
  - layers.24.self_attn.k_proj.weight
  - layers.24.self_attn.v_proj.weight
  - layers.24.self_attn.o_proj.weight
  - layers.24.self_attn.q_norm.weight
  - layers.24.self_attn.k_norm.weight
  - layers.24.mlp.gate_proj.weight
  - layers.24.mlp.up_proj.weight
  - layers.24.mlp.down_proj.weight
  - layers.24.input_layernorm.weight
  - layers.24.post_attention_layernorm.weight
  - layers.25.self_attn.q_proj.weight
  - layers.25.self_attn.k_proj.weight
  - layers.25.self_attn.v_proj.weight
  - l

## Load strand-specific gene intervals

Read the validation gene intervals for forward and reverse orientations. Each table provides genomic coordinates for a gene, including its chromosome, transcription start site (TSS), transcription end site (TES), and strand.

In [4]:
intervals_dir = '/workspace-SR003.nfs2/estsoi/scRNA_seq_experiments/GENA_LM/downstream_tasks/expression_prediction/intervals'
forward_intervals = pl.read_csv(f'{intervals_dir}/human.valid.forward.csv', separator='\t')
reverse_intervals = pl.read_csv(f'{intervals_dir}/human.valid.reverse.csv', separator='\t')

## Initialize genome access and cell-type metadata

Create a `Genome` object for retrieving DNA and genomic annotations from the reference genome.

Create a `DescriptionLookup` from the selected metadata files. Each identifier represents one cell type or experimental condition whose description will be supplied to the model.

In [5]:
genome = Genome(
    fasta_path=GENOME_FASTA,
    annotation=GENOME_ANNOTATION,
)
lookup = DescriptionLookup(
    json_dir=DESCRIPTION_DIR,
    keys=['ENCFF035CWS',
            'ENCFF083EOC',
            'ENCFF123KIW',
            'ENCFF236XOK',
            'ENCFF242BWW',
            'ENCFF329ENM',
            'ENCFF361XCF',
            'ENCFF494KRC',
            'ENCFF602HCV',
            'ENCFF660EXG',
            'ENCFF664WLU',
            'ENCFF761SPP',
            'ENCFF784MDF',
            'ENCFF857JQM']
)

/workspace-SR003.nfs2/estsoi/API/GENA_LM/downstream_tasks/expression_prediction/api/src/gena_expression/genome.py:423: RuntimeWarning: Plain GTF/GFF annotations are not supported directly for fast interval queries; using existing compressed annotation /workspace-SR003.nfs2/estsoi/scRNA_seq_experiments/GENA_LM/downstream_tasks/expression_prediction/datasets/data/genomes/hg38/gencode.v29.primary_assembly.annotation_UCSC_names.gtf.gz. debug={'plain_path': '/workspace-SR003.nfs2/estsoi/scRNA_seq_experiments/GENA_LM/downstream_tasks/expression_prediction/datasets/data/genomes/hg38/gencode.v29.primary_assembly.annotation_UCSC_names.gtf', 'compressed_path': '/workspace-SR003.nfs2/estsoi/scRNA_seq_experiments/GENA_LM/downstream_tasks/expression_prediction/datasets/data/genomes/hg38/gencode.v29.primary_assembly.annotation_UCSC_names.gtf.gz'}
  return _TabixGtfAnnotation(cls._ensure_tabix_gtf(path))
/workspace-SR003.nfs2/estsoi/API/GENA_LM/downstream_tasks/expression_prediction/api/src/gena_expr

## Build conditions and coordinate lookup tables

Convert every metadata entry into a `Condition` object containing a human-readable name and its description.

Create dictionaries mapping each gene identifier to its chromosome, TSS, TES, and strand separately for forward and reverse interval sets. These dictionaries are used to generate the corresponding DNA inputs efficiently.

In [6]:
cellType2condition = {
    cell_type: Condition(name=cell_type, description=lookup[cell_type])
    for cell_type in lookup.keys()
}
tss_tes_coords_forward = {row['gene_id'] : (row['chrom'], row['TSS'], row['TES'], row['gene_strand']) for row in forward_intervals.iter_rows(named=True)}
tss_tes_coords_reverse = {row['gene_id'] : (row['chrom'], row['TSS'], row['TES'], row['gene_strand']) for row in reverse_intervals.iter_rows(named=True)}

## Fetch one annotated sequence around a gene

Define a worker function that retrieves the genomic DNA sequence for one gene. The requested interval includes a fixed amount of context upstream of the TSS in the model’s reading direction.

The returned sequence retains genomic features and receives an explicit `tss` feature. This feature is later used to align inference windows at the transcription start site.

In [7]:
def _get_sequence(item, genome, reverse, token_len_for_fetch, num_before):
    gene, (chrom, tss, tes, strand) = item

    start, end = (
        (tes, tss + token_len_for_fetch * num_before)
        if reverse
        else (tss - token_len_for_fetch * num_before, tes)
    )

    return genome.sequence(
        chrom=chrom,
        start=start,
        end=end,
        strand=strand,
        include_features=True,
        name=gene,
    ).add_feature(name='tss',
                    type='tss', 
                    start=token_len_for_fetch*num_before, 
                    end=token_len_for_fetch*num_before)

## Generate sequence–condition pairs in parallel

Define a helper that retrieves all gene sequences with multiple worker processes and then pairs every gene with every requested cell type.

For each cell type, the same ordered gene sequence collection is reused while the associated `Condition` changes. The result is a complete Cartesian product of genes and cell-type descriptions, ready for batched inference.

In [8]:
def make_sequences_and_conditions(
    genome: Genome,
    cell_types:list|tuple,
    cellType2condition:dict,
    tss_tes_coords,
    reverse:bool,
    max_workers:int,
    num_before:int,
    token_len_for_fetch:int
):
    cell_types = list(cell_types)
    coordinate_items = list(tss_tes_coords.items())

    worker = partial(
        _get_sequence,
        genome=genome,
        reverse=reverse,
        num_before=num_before, 
        token_len_for_fetch=token_len_for_fetch
    )

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        gene_sequences = list(
            tqdm.tqdm(
                executor.map(
                    worker,
                    coordinate_items,
                    chunksize=100,
                ),
                total=len(coordinate_items),
                desc="Fetching sequences",
            )
        )

    sequences = gene_sequences * len(cell_types)

    conditions = [
        cellType2condition[cell_type]
        for cell_type in cell_types
        for _ in gene_sequences
    ]

    return sequences, conditions

## Build forward- and reverse-orientation inputs

Generate annotated gene sequences for the forward and reverse interval sets and pair them with all cell-type conditions.

Parallel sequence retrieval substantially reduces preprocessing time for large gene collections.

In [9]:
sequences_forward, conditions_forward = make_sequences_and_conditions(genome=genome,
                                                            cell_types=lookup.keys(),
                                                            cellType2condition=cellType2condition,
                                                            tss_tes_coords=tss_tes_coords_forward,
                                                            reverse=False,
                                                            max_workers=10,
                                                            num_before=512,
                                                            token_len_for_fetch=15
                                                            )

sequences_reverse, conditions_reverse = make_sequences_and_conditions(genome=genome,
                                                            cell_types=lookup.keys(),
                                                            cellType2condition=cellType2condition,
                                                            tss_tes_coords=tss_tes_coords_reverse,
                                                            reverse=True,
                                                            max_workers=10,
                                                            num_before=512,
                                                            token_len_for_fetch=15
                                                            )

Fetching sequences: 100%|██████████| 1490/1490 [00:03<00:00, 461.05it/s]


## Reference implementation: serial sequence construction

This cell shows the equivalent nested-loop implementation for constructing forward and reverse inputs without parallel processing.

It is retained as a transparent reference for the input construction logic, but is slower than `make_sequences_and_conditions` for large datasets.

In [ ]:
#SLOW, same as code above, DO NOT RUN!
conditions_forward=[]
sequences_forward = []
for cell_type in lookup.keys():
    for gene, (chrom, tss, tes, strand) in tqdm.tqdm(tss_tes_coords_forward.items()):
        sequence = genome.sequence(
                chrom=chrom,
                start=tss-15*512, #token_len_for_fetch * num_before
                end=tes,
                strand=strand,
                include_features=True,
                name=gene,
            )
        sequences_forward.append(sequence)
        conditions_forward.append(cellType2condition[cell_type])

conditions_reverse=[]
sequences_reverse = []
for cell_type in lookup.keys():
    for gene, (chrom, tss, tes, strand) in tqdm.tqdm(tss_tes_coords_reverse.items()):
        sequence = genome.sequence(
                chrom=chrom,
                start=tes, 
                end=tss+15*512, #token_len_for_fetch * num_before
                strand=strand, # strand == "-" would take reverse complement
                include_features=True,
                name=gene,
            ).add_feature(name='tss',
                    type='tss', 
                    start=15*512, 
                    end=15*512)
        sequences_reverse.append(sequence)
        conditions_reverse.append(cellType2condition[cell_type])

100%|██████████| 1490/1490 [00:03<00:00, 436.12it/s]


## Combine all inference inputs

Concatenate forward and reverse gene sequences, along with their matching conditions, into one collection for model inference.

In [10]:
sequences = sequences_forward + sequences_reverse
conditions = conditions_forward + conditions_reverse

## Predict expression for every gene–condition pair

Run batched expression prediction across all prepared sequences and cell-type conditions.

Predictions are centered on the explicitly annotated `tss` feature. Each prediction is converted into a compact record containing the cell type, gene identifier, and scalar expression value.

In [11]:
results = []
predictions = model.predict_multiple_sequences(
    sequences=sequences,
    conditions=conditions,
    center='tss',
    grouping='no_grouping',
    return_tokens=True,
    preprocessing_workers=10,
    prefetch_batches=2,
    max_records_per_forward=400
)
for prediction in predictions:
    results.append(
        (
            prediction.condition.name,
            prediction.sequence.name,
            float(prediction.outputs["expression"].flatten()[0]),
        )
    )

Predicting sequences: 100%|██████████| 107/107 [05:01<00:00,  2.82s/batch]


## Create a gene-by-cell-type prediction matrix

Convert the prediction records into a tabular data frame, then pivot it so that rows represent genes and columns represent cell types.

This matrix format matches the layout used for the ground-truth expression measurements.

In [12]:
df_predicted = pl.DataFrame(results, schema={'Cell Type':pl.String, 'Gene':pl.String, 'Expression':pl.Float64})
df_predicted.columns = ['Cell Type', 'Gene', 'Expression']
df_predicted = df_predicted.pivot(on='Cell Type', index='Gene')

/tmp/ipykernel_1008676/894603704.py:1: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  df_predicted = pl.DataFrame(results, schema={'Cell Type':pl.String, 'Gene':pl.String, 'Expression':pl.Float64})


In [20]:
df_predicted.sort(by='Gene')

Gene,ENCFF035CWS,ENCFF083EOC,ENCFF123KIW,ENCFF236XOK,ENCFF242BWW,ENCFF329ENM,ENCFF361XCF,ENCFF494KRC,ENCFF602HCV,ENCFF660EXG,ENCFF664WLU,ENCFF761SPP,ENCFF784MDF,ENCFF857JQM
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""ENSG00000001617.11""",0.5078125,1.6640625,1.5703125,0.722656,1.4609375,1.9765625,1.9140625,2.171875,2.0625,1.1953125,1.734375,1.0,0.404297,2.046875
"""ENSG00000002016.17""",0.194336,0.116211,0.103027,0.163086,0.12793,0.092285,0.214844,0.204102,0.390625,0.259766,0.111328,0.333984,0.146484,0.158203
"""ENSG00000002549.12""",2.9375,2.859375,2.890625,3.140625,3.0,2.578125,3.4375,3.203125,3.3125,3.28125,3.1875,3.671875,3.046875,2.765625
"""ENSG00000002587.9""",1.1796875,0.6484375,0.921875,0.746094,0.5078125,0.71875,0.486328,0.3671875,0.211914,0.155273,0.597656,0.128906,0.198242,0.839844
"""ENSG00000003393.14""",2.28125,2.0625,2.109375,2.140625,2.015625,2.0625,1.8515625,1.71875,2.328125,2.46875,2.109375,2.578125,2.109375,2.203125
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000285901.1""",1.328125,0.9921875,1.1171875,1.3203125,1.4375,1.0859375,1.4296875,0.972656,1.140625,0.6796875,0.761719,0.503906,0.558594,1.3828125
"""ENSG00000285971.1""",1.078125,0.458984,0.5390625,0.578125,0.46875,0.496094,0.261719,0.183594,0.263672,0.169922,0.292969,0.103516,0.207031,0.6015625
"""ENSG00000285972.1""",0.008972,0.013916,0.462891,-0.001495,0.012085,0.033936,0.015137,-0.003693,0.061523,0.125977,-0.007507,0.136719,0.108398,0.025635


In [16]:
ground_true = pl.read_csv('/workspace-SR003.nfs2/estsoi/scRNA_seq_experiments/v1_ground_true.csv')

In [21]:
ground_true.sort('gene_id')

gene_id,ENCFF035CWS,ENCFF083EOC,ENCFF123KIW,ENCFF236XOK,ENCFF242BWW,ENCFF329ENM,ENCFF361XCF,ENCFF494KRC,ENCFF602HCV,ENCFF660EXG,ENCFF664WLU,ENCFF761SPP,ENCFF784MDF,ENCFF857JQM
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""ENSG00000001617.11""",1.079541,1.109003,2.621794,1.266682,3.045891,0.540609,2.526034,1.884014,3.266901,2.67877,1.053421,2.218972,1.311255,2.637604
"""ENSG00000002016.17""",2.286177,2.088097,1.595354,1.543407,1.178615,2.598605,0.909459,2.293584,1.871749,1.846946,2.037829,2.124429,2.028351,1.698534
"""ENSG00000002549.12""",3.037178,2.973722,3.893732,3.832327,3.886906,3.097516,3.550817,2.588759,2.945033,3.730597,2.922421,3.898263,3.879168,4.093605
"""ENSG00000002587.9""",0.546619,0.256456,2.593309,0.687508,1.325394,0.918205,1.041303,0.168223,0.162103,0.006797,0.333568,0.025348,0.050543,0.348569
"""ENSG00000003393.14""",2.238377,1.759425,2.019705,1.69556,1.892243,1.386311,1.919276,1.646933,2.870818,2.422747,1.912536,2.148924,2.786094,3.025538
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000285901.1""",0.724827,0.195678,0.240694,0.637291,0.688993,0.467591,0.007235,0.002128,0.708695,0.754854,0.35787,0.252319,0.234298,0.022656
"""ENSG00000285971.1""",0.184287,0.015292,0.001311,0.116774,0.001219,0.008034,0.007235,0.002128,0.039518,0.006797,0.004869,0.025348,0.00378,0.022656
"""ENSG00000285972.1""",0.014966,0.015292,0.256666,0.004663,0.012511,0.196246,0.007235,0.002128,0.055581,0.05551,0.004869,0.295898,0.029174,0.022656


## Evaluate correlation across genes within each cell type

For every shared cell type, calculate the Pearson correlation between predicted and measured expression across genes.

This metric measures how well the model ranks or separates genes within a particular cellular context.

In [18]:
pred = df_predicted.to_pandas().set_index("Gene")
true = ground_true.to_pandas().set_index("gene_id")

common_genes = true.index.intersection(pred.index)
common_cells = true.columns.intersection(pred.columns)

true = true.loc[common_genes, common_cells]
pred = pred.loc[common_genes, common_cells]

rows = []

for cell in common_cells:
    true_vec = true[cell].astype(float).values
    pred_vec = pred[cell].astype(float).values
    corr = np.corrcoef(true_vec, pred_vec)[0, 1]
    rows.append({
        "cell_type": cell,
        "corr_genes": corr,
    })

result = pd.DataFrame(rows)

## Evaluate correlation across cell types within each gene

For every shared gene, calculate the Pearson correlation between predicted and measured expression across cell types.

Genes with too few valid observations or no variation in their measured expression are excluded because correlation is undefined or uninformative in those cases.

In [19]:
gene_corrs = []
skipped_genes = []
for gene in common_genes:
    gene_true = true.loc[gene].astype(float)
    gene_pred = pred.loc[gene].astype(float)

    mask = pd.notna(gene_true) & pd.notna(gene_pred)

    gene_true = gene_true[mask]
    gene_pred = gene_pred[mask]

    if len(gene_true) > 3 and np.std(gene_true) > 0:
        corr = np.corrcoef(gene_true, gene_pred)[0, 1]

        if not np.isnan(corr):
            gene_corrs.append(corr)
        else:
            skipped_genes.append(gene)
    else:
        skipped_genes.append(gene)

mean_corr_cells = np.mean(gene_corrs)
result["corr_cells"] = mean_corr_cells

mean_row = pd.DataFrame([{
    "cell_type": "mean",
    "corr_genes": result["corr_genes"].mean(),
    "corr_cells": mean_corr_cells,
}])

result = pd.concat([result, mean_row], ignore_index=True)

result

,cell_type,corr_genes,corr_cells
0,ENCFF035CWS,0.790096,0.28662
1,ENCFF083EOC,0.748865,0.28662
2,ENCFF123KIW,0.768191,0.28662
3,ENCFF236XOK,0.771661,0.28662
4,ENCFF242BWW,0.754279,0.28662
5,ENCFF329ENM,0.761403,0.28662
6,ENCFF361XCF,0.793126,0.28662
7,ENCFF494KRC,0.747706,0.28662
8,ENCFF602HCV,0.819241,0.28662
9,ENCFF660EXG,0.781626,0.28662
